# Coffee Bean Index — Hermes-3 Fine-tune
**Phase 2 of 4: Fine-tuning run on Google Colab (T4 GPU)**

## Before you start
1. Runtime → Change runtime type → **T4 GPU** (free tier)
2. Upload `train.jsonl` and `val.jsonl` from `training_data/finetune/` to your Google Drive at:  
   `My Drive/coffee-finetune/data/train.jsonl`  
   `My Drive/coffee-finetune/data/val.jsonl`
3. Run cells top to bottom — do not skip any.

**Expected runtime:** 2–4 hours on T4. Do not close the tab.  
**Output:** LoRA adapter saved to `My Drive/coffee-finetune/models/hermes-coffee-v1/`

In [ ]:
# Install — takes 3-5 minutes. Ignore dependency warnings.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "unsloth",
    "trl>=0.8.6",
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "accelerate>=0.29.0",
    "bitsandbytes>=0.43.0",
    "xformers",
], check=False)

print("Install complete.")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1e9
print(f"GPU: {gpu.name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 12:
    raise RuntimeError(f"Only {vram_gb:.1f} GB VRAM — need at least 12 GB (T4 = 15 GB).")

print("GPU check passed.")

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

DATA_DIR   = "/content/drive/MyDrive/coffee-finetune/data"
OUTPUT_DIR = "/content/drive/MyDrive/coffee-finetune/models/hermes-coffee-v1"
CKPT_DIR   = "/content/drive/MyDrive/coffee-finetune/checkpoints"

for d in [DATA_DIR, OUTPUT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
VAL_FILE   = os.path.join(DATA_DIR, "val.jsonl")

for path in [TRAIN_FILE, VAL_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Upload train.jsonl and val.jsonl to My Drive/coffee-finetune/data/"
        )

print(f"train.jsonl: {os.path.getsize(TRAIN_FILE):,} bytes")
print(f"val.jsonl:   {os.path.getsize(VAL_FILE):,} bytes")
print("Drive mounted and files found.")

In [ ]:
import json
from collections import Counter

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  WARNING: skip line {i+1}: {e}")
    return records

train_raw = load_jsonl(TRAIN_FILE)
val_raw   = load_jsonl(VAL_FILE)

print(f"Train records: {len(train_raw)}")
print(f"Val records:   {len(val_raw)}")

# Validate structure
for split_name, records in [("train", train_raw), ("val", val_raw)]:
    bad = [i for i, r in enumerate(records)
           if "messages" not in r or len(r["messages"]) < 3]
    if bad:
        print(f"  WARNING: {len(bad)} malformed records in {split_name} at indices {bad[:5]}")

# Category breakdown
cats = Counter()
for r in train_raw:
    cats[r.get("meta", {}).get("category", "unknown")] += 1
print("\nTraining categories:")
for cat, count in cats.most_common():
    print(f"  {cat}: {count}")

# Preview first record
print("\nSample (train[0]):")
sample = train_raw[0]
for msg in sample["messages"]:
    preview = msg["content"][:120].replace("\n", " ")
    print(f"  [{msg['role']}] {preview}...")

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "NousResearch/Hermes-3-Llama-3.1-8B"

print(f"Loading {MODEL_NAME} with 4-bit quantization...")
print("This downloads ~4.5 GB — takes 5-10 minutes on first run.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,   # auto: bfloat16 on Ampere+, float16 on T4
    load_in_4bit   = True,   # QLoRA: fits 8B in 15 GB VRAM
)

print(f"\nModel loaded. Tokenizer vocab size: {tokenizer.vocab_size:,}")
print(f"EOS token: {repr(tokenizer.eos_token)} (id={tokenizer.eos_token_id})")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    lora_alpha         = 32,
    lora_dropout       = 0.05,
    target_modules     = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias               = "none",
    use_gradient_checkpointing = "unsloth",   # saves VRAM
    random_state       = 42,
    use_rslora         = False,
)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100*trainable/total:.2f}% of {total:,})")

In [ ]:
from datasets import Dataset

EOS = tokenizer.eos_token

def format_chatml(record):
    """Convert messages list to ChatML string with EOS token appended."""
    text = ""
    for msg in record["messages"]:
        text += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    text += EOS
    return {"text": text}

train_formatted = [format_chatml(r) for r in train_raw]
val_formatted   = [format_chatml(r) for r in val_raw]

train_dataset = Dataset.from_list(train_formatted)
val_dataset   = Dataset.from_list(val_formatted)

# Sanity check lengths
import numpy as np
lengths = [len(tokenizer.encode(r["text"])) for r in train_formatted[:50]]
p95 = int(np.percentile(lengths, 95))
print(f"Token length p95 (sample of 50): {p95}")
if p95 > MAX_SEQ_LENGTH:
    print(f"  WARNING: p95 exceeds MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. Some records will be truncated.")
else:
    print(f"  OK — fits within MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}")

print(f"\nTrain dataset: {len(train_dataset)} examples")
print(f"Val dataset:   {len(val_dataset)} examples")

# Preview formatted text
print("\nFormatted sample (first 400 chars):")
print(train_formatted[0]["text"][:400])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,          # effective batch = 8
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    optim                       = "adamw_8bit",
    logging_steps               = 10,
    evaluation_strategy         = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 2,          # keep last 2 checkpoints
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",     # disable wandb
    seed                        = 42,
)

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    args             = training_args,
)

# Estimate steps
steps_per_epoch = len(train_dataset) // (
    training_args.per_device_train_batch_size *
    training_args.gradient_accumulation_steps
)
total_steps = steps_per_epoch * training_args.num_train_epochs
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps:     {total_steps}")
print(f"Eval every:      {training_args.eval_steps} steps")
print(f"Save every:      {training_args.save_steps} steps")
print("\nTrainer ready. Run the next cell to start training.")

In [ ]:
import time

print("Starting training...")
print("Loss will print every 10 steps. Eval loss at every 50 steps.")
print("Do not close this tab.\n")

start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start

print(f"\nTraining complete in {elapsed/3600:.1f} hours ({elapsed/60:.0f} minutes).")
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation results:")
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# Val loss below 1.5 is good for this corpus size
# Val loss below 1.2 is excellent
val_loss = eval_results.get("eval_loss", None)
if val_loss is not None:
    if val_loss < 1.2:
        print(f"\n✅ Excellent — val loss {val_loss:.3f} < 1.2")
    elif val_loss < 1.5:
        print(f"\n✅ Good — val loss {val_loss:.3f} < 1.5")
    elif val_loss < 2.0:
        print(f"\n⚠️  Marginal — val loss {val_loss:.3f}. Consider 1 more epoch.")
    else:
        print(f"\n❌ High val loss {val_loss:.3f}. Check data quality before serving.")

In [ ]:
print(f"Saving LoRA adapter to {OUTPUT_DIR} ...")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os, glob
saved_files = glob.glob(os.path.join(OUTPUT_DIR, "*"))
print(f"\nSaved {len(saved_files)} files:")
for f in sorted(saved_files):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f)}: {size_mb:.1f} MB")

print("\nAdapter saved. The adapter directory is what you push to Hugging Face.")
print("Do NOT push the base model weights — only the adapter files (~130 MB).")

In [ ]:
# Quick inference test to verify the adapter is working
FastLanguageModel.for_inference(model)

test_prompt = """<|im_start|>system
You are a coffee expert writing for a niche review and knowledge site. You have deep
technical knowledge of espresso, filter brewing, grinders, origins, and roast science.
Your voice is direct, confident, and specific. No hedging. No filler. No marketing language.<|im_end|>
<|im_start|>user
Write a 3-sentence verdict for Lavazza Super Crema espresso beans. Focus on flavor, who it suits, and value.<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens   = 200,
        temperature      = 0.7,
        top_p            = 0.9,
        repetition_penalty = 1.1,
        do_sample        = True,
        pad_token_id     = tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== Model output ===")
print(response.strip())
print("===================")
print("\nIf this reads like coherent coffee writing (not gibberish), the fine-tune worked.")

## Optional: Push adapter to Hugging Face Hub

Run the cell below only if you want to host the adapter on Hugging Face (needed for Together AI serving).

**Before running:**
1. Create a Hugging Face account at https://huggingface.co
2. Create a private repo named `coffee-hermes-v1`
3. Generate a write token at https://huggingface.co/settings/tokens
4. Paste the token when prompted

In [ ]:
# OPTIONAL — skip if not using Together AI or HF hosting
from huggingface_hub import HfApi, login

HF_REPO = "YOUR_HF_USERNAME/coffee-hermes-v1"   # ← change this

hf_token = input("Paste your Hugging Face write token: ").strip()
login(token=hf_token, add_to_git_credential=False)

model.push_to_hub(HF_REPO, token=hf_token, private=True)
tokenizer.push_to_hub(HF_REPO, token=hf_token, private=True)

print(f"\nPushed to https://huggingface.co/{HF_REPO}")
print("Adapter is private — only you can access it.")

## Next steps

| Phase | What | File |
|---|---|---|
| ✅ Phase 1 | Format training data | `data_pipeline/format_for_finetuning.py` |
| ✅ Phase 2 | Fine-tune run | This notebook |
| ⬜ Phase 3 | Build retrieval index (RAG) | `data_pipeline/build_index.py` |
| ⬜ Phase 4 | Wire `generate_review.py` to fine-tuned model + RAG | `scrapers/generate_review.py` |

**After this notebook:**
1. Download the adapter from Google Drive: `My Drive/coffee-finetune/models/hermes-coffee-v1/`
2. Tell Claude you're ready for Phase 3 — it will build `build_index.py`